# Gamma sensitivity analysis

For each of the four return metrics (actor/absolute x episode/discounted return), final performance vs. `gamma`, one line per budget variant (5 fixed budgets + the adaptive[1-5] budget), with a shaded 95% CI band across seeds.

`absolute` is the post-training "absolute metric" (see `stoix/evaluator.py`): the best checkpoint re-evaluated for 10x as many episodes as a single eval step, logged once per run rather than as a training curve - a much less noisy final-performance estimate than the per-checkpoint `evaluator/*` curve, so it's used here in place of `evaluator/*`.

Requires a wandb cache CSV fetched with the updated `fetch_wandb_lightsout_hd64.py` (the one that exports `gamma` and `absolute/*` columns) and spanning more than one `gamma` value - currently only `wandb_cache_hd64-lightsout-icot-final.csv` (Transformer-CoT, gamma in {0.99, 0.995, 0.999, 0.9995}) qualifies.

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_palette("colorblind")

CI95_Z = 1.96  # normal-approximation 95% CI half-width, in units of SEM
N_TAIL_EVALS = 3  # matches compute_final_values' n_tail elsewhere in this repo

CSV_PATH = "wandb_cache_hd64-lightsout-icot-final.csv"
ARCH = None  # None = use whichever single arch is present in the CSV

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df[df["state"] == "finished"]
df = df[~df["sgh"]]

missing = [c for c in ("gamma", "absolute/episode_return/mean") if c not in df.columns]
if missing:
    raise ValueError(
        f"{CSV_PATH} is missing {missing} - re-fetch it with the updated fetch_wandb_lightsout_hd64.py"
    )
df = df[df["gamma"].notna()]

if ARCH is None:
    archs = df["arch"].unique()
    if len(archs) != 1:
        raise ValueError(f"Multiple architectures present ({list(archs)}) - set ARCH explicitly")
    ARCH = archs[0]
df = df[df["arch"] == ARCH]

gammas = sorted(df["gamma"].unique())
print(f"arch={ARCH}, gammas={gammas}, seeds={df['seed'].nunique()}")

In [ ]:
def budget_label(min_steps, max_steps):
    return f"uniform={min_steps}" if min_steps == max_steps else f"adaptive[{min_steps}-{max_steps}]"


def compute_final_values(sub, metric, n_tail=N_TAIL_EVALS):
    """Per-run final performance: mean of the last n_tail eval points, plus
    the run's budget/gamma so it can be grouped on afterward."""
    tail = sub.sort_values("eval_idx").groupby("run_id").tail(n_tail)
    return (
        tail.groupby("run_id")
        .agg(
            value=(metric, "mean"),
            min_steps=("min_steps", "first"),
            max_steps=("max_steps", "first"),
            gamma=("gamma", "first"),
        )
        .reset_index()
    )


def gamma_curves(sub, metric):
    """{budget_label: DataFrame[gamma, mean, ci]} for one metric, across
    every (min_steps, max_steps) budget present in `sub`."""
    final = compute_final_values(sub, metric)
    curves = {}
    budgets = sorted(
        final[["min_steps", "max_steps"]].drop_duplicates().itertuples(index=False, name=None),
        key=lambda mm: (mm[0] != mm[1], mm[0], mm[1]),
    )
    for mn, mx in budgets:
        g = final[(final["min_steps"] == mn) & (final["max_steps"] == mx)]
        stats = (
            g.groupby("gamma")["value"]
            .agg(mean="mean", std="std", n="count")
            .reset_index()
            .sort_values("gamma")
        )
        stats["ci"] = (CI95_Z * stats["std"] / np.sqrt(stats["n"])).fillna(0.0)
        curves[budget_label(mn, mx)] = stats
    return curves

In [ ]:
METRIC_PANELS = [
    ("actor", "actor/episode_return/mean", "Episode return"),
    ("actor", "actor/episode_discounted_return/mean", "Discounted return"),
    ("absolute", "absolute/episode_return/mean", "Episode return"),
    ("absolute", "absolute/episode_discounted_return/mean", "Discounted return"),
]

all_budget_labels = sorted(
    {
        budget_label(mn, mx)
        for mn, mx in df[["min_steps", "max_steps"]].drop_duplicates().itertuples(index=False)
    },
    key=lambda b: (b.startswith("adaptive"), b),
)
palette = dict(zip(all_budget_labels, sns.color_palette("colorblind", n_colors=len(all_budget_labels))))

fig, axes = plt.subplots(1, 4, figsize=(24, 5), sharex=True)

for ax, (prefix, metric, label) in zip(axes, METRIC_PANELS):
    curves = gamma_curves(df, metric)
    for budget, stats in curves.items():
        color = palette[budget]
        ax.plot(stats["gamma"], stats["mean"], marker="o", label=budget, color=color)
        ax.fill_between(
            stats["gamma"], stats["mean"] - stats["ci"], stats["mean"] + stats["ci"], color=color, alpha=0.2
        )
    ax.set_title(f"{prefix.capitalize()}: {label}", fontsize=11)
    ax.set_xlabel("gamma")
    ax.set_xticks(gammas)
    ax.set_xticklabels([f"{g:g}" for g in gammas], rotation=45, ha="right")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Final performance (95% CI across seeds)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(all_budget_labels), bbox_to_anchor=(0.5, -0.1), fontsize=9)
fig.suptitle(f"{ARCH}: gamma sensitivity", y=1.04, fontsize=13)
fig.tight_layout()
plt.show()